# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [ ]:
!pip cache purge

In [ ]:
# 1. Restart your kernel first
# 2. Use a relaxed version constraint to let pip resolve compatible sub-packages
!pip install langchain==0.2.16 langchain-community==0.2.16 langchain-openai langchain-huggingface docarray ragas

### Example 1

#### Create our QandA application

In [ ]:
from langchain.chains import RetrievalQA, LLMChain  # Not langchain_classic
from langchain.indexes import VectorstoreIndexCreator
from langchain_openai import ChatOpenAI, OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_community.vectorstores import DocArrayInMemorySearch

In [ ]:
file = '/content/OutdoorClothingCatalog_1000(2).csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [ ]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})
).from_loaders([loader])




In [ ]:

vectorstore = index.vectorstore  # Get the vectorstore
retriever = vectorstore.as_retriever()  # Get the retriever

In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [ ]:
llm = ChatOpenAI(temperature = 0.0, openai_api_key=OPENAI_API_KEY)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever(),
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

#### Coming up with test datapoints

In [ ]:
data[10]

In [ ]:
data[11]

#### Hard-coded examples

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)

# Define the output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Create the output parser
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        # Split the response to get the answer
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

# Initialize the LLM
# llm = OpenAI()
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY)

# Create the LLMChain
llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=AnswerOutputParser()
)

# Example query
query = "Is the Cozy Comfort Pullover Set available in different colors?"

# Run the chain
result = llm_chain.run({"query": query})

# Print the result
print(result)


#### LLM-Generated examples

In [ ]:
import langchain.evaluation.qa
print(dir(langchain.evaluation.qa))

In [ ]:
from langchain.evaluation.qa import QAGenerateChain

In [ ]:
# 1. Initialize the LLM
llm = ChatOpenAI(model="gpt-4o", openai_api_key=OPENAI_API_KEY) # Use your preferred model

# 2. Create the generation chain
example_gen_chain = QAGenerateChain.from_llm(llm)

In [ ]:
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

In [ ]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

In [ ]:
d_flattened = [data['qa_pairs'] for data in new_examples]
d_flattened

#### Combine examples

In [ ]:
examples

In [ ]:
# examples += new_example
examples += d_flattened

In [ ]:
examples[0]

In [ ]:
qa.invoke(examples[0]["query"])

### Manual Evaluation - Fun part

In [ ]:
import langchain
langchain.debug = True

# qa needs 'query' as input, and it handles context internally
result = qa.invoke({
    "query": examples[0]["query"]
})
print(result)

In [ ]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [ ]:
# When creating the RetrievalQA chain, add return_source_documents=True
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True  # Add this
)

In [ ]:
predictions = []
for i, eg in enumerate(examples[:5]):
    result = qa.invoke({"query": eg["query"]})

    pred_dict = {
        'query': eg["query"],
        'answer': eg["answer"],
        'result': result['result'],
        'source_documents': result.get('source_documents', [])
    }
    predictions.append(pred_dict)

for i, eg in enumerate(examples[:5]):
    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real Answer:", predictions[i]['answer'])
    print("Predicted Answer:", predictions[i]['result'])
    print(f"Used {len(predictions[i]['source_documents'])} source documents")
    print()

LLM as a JUdge

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser  # Add this import

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0,openai_api_key=OPENAI_API_KEY)

# Simple eval prompt (no classes)
eval_prompt = PromptTemplate(
    input_variables=["query", "answer", "result"],
    template="""Score PREDICTED vs ANSWER for QUERY (0-1, higher=better match).

Query: {query}
Answer: {answer}
Predicted: {result}

Score:"""
)

eval_chain = eval_prompt | llm | StrOutputParser()  # Now this will work

# Your EXACT workflow (no classes)
graded_outputs = []
for i, eg in enumerate(predictions):
    score_raw = eval_chain.invoke({
        "query": predictions[i]['query'],
        "answer": predictions[i]['answer'],
        "result": predictions[i]['result']
    })
    score = float(score_raw.strip())  # Extract number

    graded_outputs.append({'score': score})

    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real:", predictions[i]['answer'])
    print("Predicted:", predictions[i]['result'])
    print("Score:", score, "\n")

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
loader = TextLoader("/content/nyc_text.txt")
index = VectorstoreIndexCreator(embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cuda'})).from_loaders([loader])

# Explicitly define the retriever
retriever = index.vectorstore.as_retriever()

llm = ChatOpenAI(temperature= 0, openai_api_key=OPENAI_API_KEY)
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    return_source_documents=True,
)

In [ ]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain.invoke({"query": question})
result["result"]

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [ ]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [ ]:
# Generate predictions with your qa_chain
all_predictions = []
for example in examples:
    q = example["query"]
    result = qa_chain.invoke({"query": q})  # RetrievalQA uses 'query'

    all_predictions.append({
        "question": q,
        "answer": result['result'],  # Extract answer from result dict
        "contexts": [doc.page_content for doc in result['source_documents']],  # Extract contexts
        "ground_truth": example["ground_truths"][0]
    })

print("Predictions generated:", len(all_predictions))

# Display first prediction to verify
print("\nExample prediction:")
print("Question:", all_predictions[0]['question'])
print("Answer:", all_predictions[0]['answer'])
print("Contexts:", len(all_predictions[0]['contexts']), "documents")
print("Ground Truth:", all_predictions[0]['ground_truth'])

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

evaluate(): evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).

In [ ]:
#evaluate(): evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).

import os
from google.colab import userdata

# Set OpenAI API key as environment variable
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Now run RAGAS evaluation
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

dataset_all = Dataset.from_list(all_predictions)

print("=" * 50)
print("RAGAS Evaluation - Faithfulness")
print("=" * 50)
f = evaluate(dataset_all, metrics=[faithfulness])
print(f)

print("\n" + "=" * 50)
print("RAGAS Evaluation - Context Recall")
print("=" * 50)
r = evaluate(dataset_all, metrics=[context_recall])
print(r)

print("\n" + "=" * 50)
print("RAGAS Evaluation - Combined Metrics")
print("=" * 50)
results = evaluate(dataset_all, metrics=[faithfulness, context_recall])
print(results)

In [ ]:
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

result_updated = {}
for old_key, new_key in key_mapping.items():
    if old_key in result:
        result_updated[new_key] = result[old_key]

In [ ]:
print("=" * 70)
print("RAGAS EVALUATION WITH EvaluatorChain")
print("=" * 70)

from ragas.integrations.langchain import EvaluatorChain
from ragas.metrics import faithfulness, context_recall, answer_relevancy

# Create evaluation chains
print("\n1. Creating evaluation chains...")
faithfulness_chain = EvaluatorChain(metric=faithfulness)
context_recall_chain = EvaluatorChain(metric=context_recall)

print("✓ Evaluation chains created")

Method 1: Evaluate a Single Result with __call__()

In [ ]:
#Method 1: Evaluate a Single Result with call()

print("\n" + "=" * 70)
print("METHOD 1: Evaluate Single Result with __call__()")
print("=" * 70)

# Get a single result from the QA chain
single_query = examples[0]["query"]
print(f"\nQuery: {single_query}")

result = qa_chain.invoke({"query": single_query})
print(f"Answer: {result['result']}")

# Prepare the result for Ragas EvaluatorChain
single_eval_input = {
    "question": result["query"],
    "answer": result["result"],
    "contexts": [doc.page_content for doc in result["source_documents"]],
    "ground_truth": examples[0]["ground_truths"][0] # Ensure ground_truth is singular and a single string
}

# Evaluate faithfulness
print("\n--- Evaluating Faithfulness ---")
faithfulness_result = faithfulness_chain(single_eval_input)
print(f"Faithfulness Score: {faithfulness_result['faithfulness']:.3f}") # Corrected key

# Evaluate context recall
print("\n--- Evaluating Context Recall ---")
context_recall_result = context_recall_chain(single_eval_input) # Pass the same correctly formatted input
print(f"Context Recall Score: {context_recall_result['context_recall']:.3f}") # Corrected key

Test with Fake Results (Low Scores)

In [ ]:
#Test with Fake Results (Low Scores)

print("\n" + "=" * 70)
print("TESTING WITH FAKE RESULTS (Expected Low Scores)")
print("=" * 70)

# The base input structure for EvaluatorChain, derived from result and examples[0]
# 'result' and 'examples[0]' are available from previous cell execution
base_eval_input_for_test = {
    "question": result["query"],
    "answer": result["result"],
    "contexts": [doc.page_content for doc in result["source_documents"]],
    "ground_truth": examples[0]["ground_truths"][0]
}

# Test 1: Fake answer (low faithfulness)
print("\n--- Test 1: Fake Answer ---")
fake_faithfulness_input = base_eval_input_for_test.copy()
fake_faithfulness_input["answer"] = "The population of NYC is 100 million people and it's on Mars."
fake_faithfulness = faithfulness_chain(fake_faithfulness_input)
print(f"Original Answer Faithfulness: {faithfulness_result['faithfulness']:.3f}")
print(f"Fake Answer Faithfulness: {fake_faithfulness['faithfulness']:.3f}")
print("→ Lower score because answer contradicts the source documents")

# Test 2: Fake source documents (low context recall)
print("\n--- Test 2: Fake Source Documents ---")
from langchain.schema import Document
fake_contexts_docs = [
    Document(page_content="I love pizza and ice cream."),
    Document(page_content="The weather is nice today.")
]
fake_context_recall_input = base_eval_input_for_test.copy()
fake_context_recall_input["contexts"] = [doc.page_content for doc in fake_contexts_docs]
# The ground_truth should remain the original one for context recall evaluation
fake_context_recall = context_recall_chain(fake_context_recall_input)
print(f"Original Context Recall: {context_recall_result['context_recall']:.3f}")
print(f"Fake Context Recall: {fake_context_recall['context_recall']:.3f}")
print("→ Lower score because ground truth is not in source documents")

Method 2: Batch Evaluate with evaluate()

In [ ]:
#Method 2: Batch Evaluate with evaluate()

def evaluate_qa_pipeline(
    qa_chain,
    examples,
    evaluators,
    prediction_key="result",
    context_key="source_documents",
):
    print("\n" + "=" * 70)
    print("RUNNING BATCH EVALUATION")
    print("=" * 70)

    # Generate predictions
    predictions = qa_chain.batch([{"query": ex["query"]} for ex in examples])

    def extract_contexts(pred):
        docs = pred.get(context_key, [])
        return [doc.page_content for doc in docs] if docs else []

    results = {"metrics": {}}

    for name, evaluator in evaluators.items():
        print(f"\n--- Evaluating {name} ---")

        eval_inputs = [
            {
                "question": ex["query"],
                "answer": pred.get(prediction_key, ""),
                "contexts": extract_contexts(pred),
                "ground_truth": ex["ground_truths"][0] if ex.get("ground_truths") else "",
            }
            for ex, pred in zip(examples, predictions)
        ]

        eval_outputs = evaluator.batch(eval_inputs)

        # Flexible score extraction
        def extract_score(res):
            if "score" in res:
                return res["score"]
            if name in res:
                return res[name]
            if "value" in res:
                return res["value"]
            return None

        scores = []
        for res in eval_outputs:
            score = extract_score(res)
            if score is not None:
                scores.append(score)
            else:
                print("Unknown format:", res)

        avg_score = sum(scores) / len(scores) if scores else 0.0

        print(f"Average {name}: {avg_score:.3f}")
        print(f"Scores: {scores}")

        results["metrics"][name] = {
            "average": avg_score,
            "scores": scores,
        }

    return results

In [ ]:
evaluators = {
    "faithfulness": faithfulness_chain,
    "context_recall": context_recall_chain,
}

results = evaluate_qa_pipeline(
    qa_chain=qa_chain,
    examples=examples,
    evaluators=evaluators,
)

SECTION 1 — Single Example Evaluation (Understanding RAGAS Basics)

In [ ]:
from ragas.integrations.langchain import EvaluatorChain
from ragas.metrics import faithfulness, context_recall

# create evaluators
faithfulness_chain = EvaluatorChain(metric=faithfulness)
context_recall_chain = EvaluatorChain(metric=context_recall)

# run one query through RAG pipeline
query = examples[0]["query"]
result = qa_chain.invoke({"query": query})

# format into RAGAS input schema
single_eval_input = {
    "question": query,
    "answer": result["result"],
    "contexts": [doc.page_content for doc in result["source_documents"]],
    "ground_truth": examples[0]["ground_truths"][0]
}

# evaluate single example
faith_score = faithfulness_chain(single_eval_input)
recall_score = context_recall_chain(single_eval_input)

print("Faithfulness:", faith_score["faithfulness"])
print("Context Recall:", recall_score["context_recall"])

SECTION 2 — Batch Evaluation (LangChain Style)

In [ ]:
def run_batch_evaluation(qa_chain, examples):
    predictions = []

    for ex in examples:
        result = qa_chain.invoke({"query": ex["query"]})

        predictions.append({
            "question": ex["query"],
            "answer": result["result"],
            "contexts": [doc.page_content for doc in result["source_documents"]],
            "ground_truth": ex["ground_truths"][0],
        })

    return predictions


predictions = run_batch_evaluation(qa_chain, examples)

In [ ]:
eval_inputs = [
    {
        "question": p["question"],
        "answer": p["answer"],
        "contexts": p["contexts"],
        "ground_truth": p["ground_truth"],
    }
    for p in predictions
]

faith_batch = faithfulness_chain.batch(eval_inputs)
recall_batch = context_recall_chain.batch(eval_inputs)

# average manually
faith_scores = [r["faithfulness"] for r in faith_batch]
recall_scores = [r["context_recall"] for r in recall_batch]

print("Avg Faithfulness:", sum(faith_scores)/len(faith_scores))
print("Avg Context Recall:", sum(recall_scores)/len(recall_scores))

SECTION 3 — RAGAS Native Evaluation (Production Method)

In [ ]:
from ragas import evaluate
from datasets import Dataset

dataset_all = Dataset.from_list([
    {
        "question": ex["query"],
        "answer": pred["answer"],
        "contexts": pred["contexts"],
        "ground_truths": ex["ground_truths"],
    }
    for ex, pred in zip(examples, predictions)
])

results = evaluate(
    dataset_all,
    metrics=[faithfulness, context_recall]
)

print(results)